# Training CSV (OpenAI)

Reads `.env` for `OPENAI_API_KEY`. Writes `classification_train.csv` (`text`, `label`).

In [ ]:
import csv
import json
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

_p = Path.cwd().resolve()
ROOT = next((a for a in [_p, *_p.parents] if (a / "pyproject.toml").is_file()), _p)
load_dotenv(ROOT / ".env")
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY in .env"

MODEL = os.environ.get("MEDFLOW_SYNTH_MODEL", "gpt-4o-mini")
LABELS = [
    "PRIOR_AUTH", "REFERRAL", "RECORDS_REQUEST",
    "LAB_RESULTS", "INSURANCE", "OTHER",
]
client = OpenAI()
rows: list[dict] = []
batch_size = 50
total = 500
for start in range(0, total, batch_size):
    n = min(batch_size, total - start)
    prompt = (
        f"Return ONLY a JSON array of {n} objects. Each object: "
        '"text" (80-200 words of realistic synthetic healthcare document excerpt), '
        f'"label" (exactly one of: {", ".join(LABELS)}). '
        "Vary clinical vs administrative tone. No PHI disclaimers."
    )
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.55,
        max_tokens=8000,
    )
    raw = r.choices[0].message.content or ""
    arr = json.loads(raw[raw.find("["): raw.rfind("]") + 1])
    for obj in arr:
        rows.append({"text": obj["text"], "label": obj["label"]})
    print(f"Batch {start}-{start+n}: ok, total rows {len(rows)}")

out = ROOT / "notebooks" / "finetune_classifier" / "classification_train.csv"
out.parent.mkdir(parents=True, exist_ok=True)
with out.open("w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["text", "label"])
    w.writeheader()
    w.writerows(rows)
print("Wrote", out, "rows", len(rows))

In [ ]:
from pathlib import Path
_p = Path.cwd().resolve()
ROOT = next((a for a in [_p, *_p.parents] if (a / "pyproject.toml").is_file()), _p)
import pandas as pd
import matplotlib.pyplot as plt

csv_path = ROOT / "notebooks" / "finetune_classifier" / "classification_train.csv"
df = pd.read_csv(csv_path)
print(df.head())
print(df["label"].value_counts())
df["label"].value_counts().plot(kind="bar", title="Class distribution")
plt.ylabel("count")
plt.tight_layout()
plt.show()